In [5]:
# Install once: import Pkg; Pkg.add("Plots")

using Random
using Plots

Random.seed!(42)

# Generate observations
true_θ = 0.70
n = 100
data = rand(n) .< true_θ

100-element BitVector:
 1
 1
 1
 1
 1
 1
 1
 0
 0
 1
 1
 1
 1
 ⋮
 1
 1
 1
 1
 0
 1
 0
 1
 0
 1
 1
 1

In [ ]:

successes = sum(data)
failures = n - successes

# Parameter grid
θ_grid = collect(range(0.001, 0.999, length=1000))

# Prior: θ ~ Beta(2, 2)
α, β = 2.0, 2.0
prior = θ_grid .^ (α - 1) .* (1 .- θ_grid) .^ (β - 1)

# Bernoulli likelihood
likelihood =
    θ_grid .^ successes .* (1 .- θ_grid) .^ failures

# Posterior ∝ likelihood × prior
posterior = prior .* likelihood
posterior ./= sum(posterior)

# Posterior summary
posterior_mean = sum(θ_grid .* posterior)

cdf = cumsum(posterior)
credible_interval = (
    θ_grid[findfirst(>=(0.025), cdf)],
    θ_grid[findfirst(>=(0.975), cdf)]
)

println("True θ: $true_θ")
println("Observed: $successes successes in $n trials")
println("Posterior mean: ", round(posterior_mean, digits=3))
println("95% credible interval: ", round.(credible_interval, digits=3))

# Scale curves for easier visual comparison
prior_plot = prior ./ maximum(prior)
likelihood_plot = likelihood ./ maximum(likelihood)
posterior_plot = posterior ./ maximum(posterior)

p1 = plot(
    θ_grid,
    prior_plot;
    title="Prior belief",
    ylabel="Relative density",
    label="Beta(2, 2)",
    color=:steelblue,
    linewidth=3,
    fillrange=0,
    fillalpha=0.2
)

p2 = plot(
    θ_grid,
    likelihood_plot;
    title="Likelihood from observed data",
    ylabel="Relative likelihood",
    label="p(data | θ)",
    color=:darkorange,
    linewidth=3,
    fillrange=0,
    fillalpha=0.2
)

p3 = plot(
    θ_grid,
    posterior_plot;
    title="Posterior belief",
    xlabel="θ",
    ylabel="Relative density",
    label="p(θ | data)",
    color=:darkgreen,
    linewidth=3,
    fillrange=0,
    fillalpha=0.2
)

vspan!(
    p3,
    collect(credible_interval);
    color=:green,
    alpha=0.15,
    label="95% credible interval"
)

vline!(
    p3,
    [posterior_mean];
    color=:black,
    linestyle=:dash,
    linewidth=2,
    label="Posterior mean"
)

vline!(
    p3,
    [true_θ];
    color=:red,
    linestyle=:dot,
    linewidth=2,
    label="True θ"
)

fig = plot(
    p1,
    p2,
    p3;
    layout=(3, 1),
    size=(850, 850),
    link=:x,
    legend=:topright
)

display(fig)
savefig(fig, "bayesian_estimation.png")